In [1]:
import os
import sys

import pandas as pd
import numpy as np
import anndata as ad
import scanpy as sc 
import tacco as tc

# The notebook expects to be executed either in the workflow directory or in the repository root folder...
sys.path.insert(1, os.path.abspath('workflow' if os.path.exists('workflow/common_code.py') else '..')) 
import common_code

In [2]:

slice_ids = ["2", "3", "4", "5", "6", "7", "9", "11", "17", "18", "19", "23", "24", "25", "26", "28", "33", "34", "36"]
def load_HMlymphNode(root_dir = '/maiziezhou_lab/Datasets/ST_datasets/humanMetastaticLymphNode/GSE251926_metastatic_lymph_node_3d.h5ad', section_id =  "1"):
    adataT = sc.read_h5ad(root_dir)
    section_id = int(section_id)  # Convert section_id to integer
    slice1 = adataT[adataT.obs['n_section'] == section_id]
    if 'gene_name' not in slice1.var.columns:
        slice1.var['gene_name'] = slice1.var_names
    slice1.obs['original_clusters'] = slice1.obs['annotation']
    slice1.obs['batch'] = section_id
    return slice1
section_ids = [4, 10]
 #----------------
reference_data = load_HMlymphNode(section_id= slice_ids[4])
puck = load_HMlymphNode(section_id= slice_ids[10])

In [4]:
import numpy as np
import scipy.sparse as sp

def make_float32(adata):
    # If it's a view, materialize it
    if adata.is_view:
        adata = adata.copy()

    # If the AnnData was opened in backed mode, reload into memory
    if getattr(adata, "isbacked", False):
        import anndata as ad
        adata = ad.read_h5ad(adata.filename, backed=None)

    # Convert X
    if sp.issparse(adata.X):
        adata.X = adata.X.asfptype()           # to float
        adata.X = adata.X.astype(np.float32)   # to float32
    else:
        adata.X = np.asarray(adata.X, dtype=np.float32)

    return adata

In [ ]:
import os.path as osp 
import scanpy as sc
rna_path = '/maiziezhou_lab2/yuling/MERFISH_spinal_cord_resolved_0718.h5ad'
rna = sc.read_h5ad(rna_path)
rows = []
 
outdir = "/maiziezhou_lab2/yuling/MouseSpinal/label_transfer/tacco/HumanLymph_output"
os.makedirs(outdir, exist_ok=True)
 
    # Subset total data for these slices
adata_subset = reference_data
puck = make_float32(puck)
adata_subset = make_float32(adata_subset)
 
# (Optional) If your counts actually live in a layer, move it into X *as float32*
if 'counts' in puck.layers:
    L = puck.layers['counts']
    puck.X = (L.asfptype().astype(np.float32) if sp.issparse(L)
            else np.asarray(L, dtype=np.float32))

if 'counts' in adata_subset.layers:
    L = adata_subset.layers['counts']
    adata_subset.X = (L.asfptype().astype(np.float32) if sp.issparse(L)
                    else np.asarray(L, dtype=np.float32))

print("puck.X dtype:", puck.X.dtype)
print("adata_subset.X dtype:", adata_subset.X.dtype)

# Force TACCO to read from X (not a layer)
import tacco as tc
tc.tl.annotate(
    puck,
    adata_subset,
    'annotation',
    result_key='ClusterName',
    counts_location='X',   # be explicit
    assume_valid_counts=True
)
import pandas as pd
tmp = puck.obsm['ClusterName']
max_col = tmp.idxmax(axis=1).rename('max_col')
max_val = tmp.max(axis=1).rename('max_val')
out = pd.concat([max_col, max_val], axis=1) 
out.index = out.index.astype(puck.obs.index.dtype)
merged = puck.obs.join(out, how='left') 
ft = merged["max_col"]
mc = merged["original_clusters"]
a = merged['original_clusters'].astype('category')
b = merged['max_col'].astype('category')

# unify (unordered) categories by taking the union
cats = a.cat.categories.union(b.cat.categories)
a = a.cat.set_categories(cats)
b = b.cat.set_categories(cats)
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(merged["max_col"], merged["original_clusters"])
num_equal = (a == b).sum()
total = len(merged)
ratio = num_equal / total

rows.append({
    "ACC": ratio,
    "ARI": ari
})
outfile = os.path.join(outdir, f"TACCO_pred.csv")
merged.to_csv(outfile)
summary = pd.DataFrame(rows)
out_csv = osp.join('/maiziezhou_lab2/yuling/MouseSpinal/label_transfer/tacco/HumanLymph_output', "TACCO_summary.csv")
summary.to_csv(out_csv)
print(summary)

puck.X dtype: float32
adata_subset.X dtype: float32
Starting preprocessing
Finished preprocessing in 1.16 seconds.
Starting annotation of data with shape (39167, 20674) and a reference of shape (38555, 20674) using the following wrapped method:
+- platform normalization: platform_iterations=0, gene_keys=annotation, normalize_to=adata
   +- multi center: multi_center=None multi_center_amplitudes=True
      +- bisection boost: bisections=4, bisection_divisor=3
         +- core: method=OT annotation_prior=None
mean,std( rescaling(gene) )  0.9029205802765075 0.24734701190420466
bisection run on 1
bisection run on 0.6666666666666667
bisection run on 0.4444444444444444
bisection run on 0.2962962962962963
bisection run on 0.19753086419753085
bisection run on 0.09876543209876543
Finished annotation in 22.39 seconds.
        ACC       ARI
0  0.624633  0.484281
